###  Libraries

In [1]:
# Import libraries
import numpy as np
import torch
import sys
from Datasets.Datasets import get_dataset
from Autoencoder import Autoencoder, CD_Autoencoder
from Visualization import Visualization
from General_Functions import General_Functions

### Global Variables

In [2]:
# Set path to module
path_to_module = './'
sys.path.append(path_to_module)

# Set device (cuda or cpu)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    torch.cuda.set_device(device)

### Visualization

In [3]:
visualization = Visualization()

### Load Dataset

In [4]:
# Set dataset_name, module_name, batch_size and n_clusters
dataset_name = "emnist_balanced_letters_A_J" # The available datasets are: emnist_balanced_digits, emnist_mnist, emnist_balanced_letters_A_J, emnist_balanced_letters_K_T, emnist_balanced_letters_U_Z, har, pendigits, waveform_v1, synthetic
batch_size = 256
n_clusters = 10

# Get dataloader, input_dim, data_np and labels
dataloader, input_dim, data_np, labels = get_dataset(dataset_name, batch_size)
print('Data_Shape is:', input_dim)
print('Batches Number is:', len(dataloader))
print(np.unique(labels))

Data_Shape is: 1
Batches Number is: 110
[10 11 12 13 14 15 16 17 18 19]


### Parameters

In [10]:
# Set latent dimension and negative slope
latent_dim = 10
negative_slope = 0

# Set pre-training epochs and learning rate
n_pret_epochs = 100
pret_lr = 1e-3

# Set lamdas, training epochs and learning rate
sil_lambda = 0.01 # 0.01
entr_lambda = 0.01 # 0.01
n_epochs = 100
lr = 5e-4

# Set use_pretrain and is_mlp_ae
use_pretrain = True # Set True for pretraining autoencoder or set False for using already pretrained weights
is_mlp_ae = False # Set True for tabular datasets e.g. pendigits or set False for image datasets e.g. emnist_mnist

### Create Autoencoder Model

In [11]:
if is_mlp_ae:
    autoencoder = Autoencoder(device=device, n_clusters=n_clusters, input_dim=input_dim, latent_dim=latent_dim, negative_slope=negative_slope)
else:
    autoencoder = CD_Autoencoder(device=device, n_clusters=n_clusters, input_dim=input_dim, latent_dim=latent_dim, negative_slope=negative_slope)

autoencoder.set_general_training_variables(dataloader=dataloader, batch_size=batch_size)
autoencoder.set_pretraining_variables(n_pret_epochs=n_pret_epochs, pret_lr=pret_lr)
autoencoder.set_training_variables(n_epochs=n_epochs, lr=lr, sil_lambda=sil_lambda, entr_lambda=entr_lambda)
autoencoder.set_path_variables(path_to_module=path_to_module, dataset_name=dataset_name)
autoencoder.set_path()
autoencoder = autoencoder.to(device)

### Pretrain Autoencoder

In [12]:
if(use_pretrain):
    autoencoder.pretrain_autoencoder()
    autoencoder.save_pretrained_weights()
else:
    model_save_path = autoencoder.data_dir_path + '/Weigths/autoencoder_weights.pth'
    autoencoder.load_state_dict(torch.load(model_save_path))

Epoch: 0, Loss: 61.279371
Epoch: 1, Loss: 38.504082
Epoch: 2, Loss: 26.839097
Epoch: 3, Loss: 18.707607
Epoch: 4, Loss: 12.860014
Epoch: 5, Loss: 8.825445
Epoch: 6, Loss: 6.184899
Epoch: 7, Loss: 4.470941
Epoch: 8, Loss: 3.402812
Epoch: 9, Loss: 2.781419
Epoch: 10, Loss: 2.441376
Epoch: 11, Loss: 2.256746
Epoch: 12, Loss: 2.151357
Epoch: 13, Loss: 2.114259
Epoch: 14, Loss: 2.073933
Epoch: 15, Loss: 2.044823
Epoch: 16, Loss: 2.034235
Epoch: 17, Loss: 2.025567
Epoch: 18, Loss: 2.000002
Epoch: 19, Loss: 1.990419
Epoch: 20, Loss: 1.970149
Epoch: 21, Loss: 1.961731
Epoch: 22, Loss: 1.950504
Epoch: 23, Loss: 1.940547
Epoch: 24, Loss: 1.936149
Epoch: 25, Loss: 1.919744
Epoch: 26, Loss: 1.922330
Epoch: 27, Loss: 1.904048
Epoch: 28, Loss: 1.885302
Epoch: 29, Loss: 1.882210
Epoch: 30, Loss: 1.869262
Epoch: 31, Loss: 1.862614
Epoch: 32, Loss: 1.861757
Epoch: 33, Loss: 1.857868
Epoch: 34, Loss: 1.830808
Epoch: 35, Loss: 1.839300
Epoch: 36, Loss: 1.812225
Epoch: 37, Loss: 1.818834
Epoch: 38, Loss: 

### Initialize the clustering layer using k-means

In [13]:
autoencoder.kmeans_initialization(n_init=100)

ACC: 0.67 PUR: 0.68 NMI: 0.63 ARI: 0.52


### Train Autoencoder

In [14]:
latent_data, labels, clustering = autoencoder.train_autoencoder()

Ep: 0 Rec L: 1.5133 Cl L: 1.1178 Entropy: 0.0000 SSil: -0.0178 ACC: 0.67 PUR: 0.67 NMI: 0.63 ARI: 0.52
Ep: 1 Rec L: 1.4969 Cl L: 1.1116 Entropy: 0.0000 SSil: -0.0116 ACC: 0.67 PUR: 0.67 NMI: 0.63 ARI: 0.53
Ep: 2 Rec L: 1.4827 Cl L: 1.1091 Entropy: 0.0000 SSil: -0.0091 ACC: 0.67 PUR: 0.67 NMI: 0.63 ARI: 0.52
Ep: 3 Rec L: 1.4959 Cl L: 1.1077 Entropy: 0.0000 SSil: -0.0077 ACC: 0.67 PUR: 0.67 NMI: 0.63 ARI: 0.52
Ep: 4 Rec L: 1.4697 Cl L: 1.1066 Entropy: 0.0000 SSil: -0.0066 ACC: 0.67 PUR: 0.67 NMI: 0.63 ARI: 0.52
Ep: 5 Rec L: 1.4714 Cl L: 1.1057 Entropy: 0.0000 SSil: -0.0057 ACC: 0.67 PUR: 0.67 NMI: 0.63 ARI: 0.52
Ep: 6 Rec L: 1.4718 Cl L: 1.1049 Entropy: 0.0000 SSil: -0.0049 ACC: 0.67 PUR: 0.67 NMI: 0.63 ARI: 0.52
Ep: 7 Rec L: 1.4643 Cl L: 1.1043 Entropy: 0.0000 SSil: -0.0043 ACC: 0.67 PUR: 0.67 NMI: 0.63 ARI: 0.52
Ep: 8 Rec L: 1.4617 Cl L: 1.1038 Entropy: 0.0000 SSil: -0.0038 ACC: 0.66 PUR: 0.66 NMI: 0.63 ARI: 0.51
Ep: 9 Rec L: 1.4572 Cl L: 1.1033 Entropy: 0.0000 SSil: -0.0033 ACC: 0.66 

### Save Clustering Results

In [ ]:
General_Functions().save_excel(autoencoder.data_dir_path, autoencoder.df_eval)

### Plot Clustering Results

In [ ]:
cluster_centers = autoencoder.get_cluster_centers().cpu().detach().numpy()
data, latent_data, labels = autoencoder.get_latent_data()
visualization.plot_tsne(latent_data, labels, clustering, cluster_centers, autoencoder.data_dir_path)